# YT crawl for audio feature analysis
by DM

In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/dongmin/Robot-Radio-Station


In [2]:
import os
import tempfile
import sys
import subprocess
from time import sleep
import argparse
from pathlib import Path
from collections import defaultdict

import math
import random

import json
import csv

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import IPython.display as ipd

from yt_dlp import YoutubeDL
import essentia.standard as es

/home/dongmin/miniconda3/envs/rrs-es/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[   INFO   ] MusicExtractorSVM: no classifier models were configured by default


In [3]:
HOME = Path.home()
CWD = Path.cwd()

In [4]:
DATASET_DIR = HOME / 'userdata' / 'dongmin' / 'smp_dataset'
DATA_DIR = DATASET_DIR / 'data'
META_DIR = CWD / 'metadata'

## ~~Preprocess~~

### Import data

In [5]:
json_paths = DATA_DIR.glob('*.json')
json_paths = sorted(json_paths, key=lambda x: int(x.stem.split('.')[-1].split('-')[0]))
json_paths = list(json_paths)

len(json_paths), json_paths[:5]

(1000,
 [PosixPath('/home/dongmin/userdata/dongmin/smp_dataset/data/mpd.slice.0-999.json'),
  PosixPath('/home/dongmin/userdata/dongmin/smp_dataset/data/mpd.slice.1000-1999.json'),
  PosixPath('/home/dongmin/userdata/dongmin/smp_dataset/data/mpd.slice.2000-2999.json'),
  PosixPath('/home/dongmin/userdata/dongmin/smp_dataset/data/mpd.slice.3000-3999.json'),
  PosixPath('/home/dongmin/userdata/dongmin/smp_dataset/data/mpd.slice.4000-4999.json')])

In [6]:
playlists = []

for j_p in json_paths:
  with open(j_p, 'r') as f:
    chunk = json.load(f)
    playlists += chunk['playlists']

len(playlists), playlists[0]

(1000000,
 {'name': 'Throwbacks',
  'collaborative': 'false',
  'pid': 0,
  'modified_at': 1493424000,
  'num_tracks': 52,
  'num_albums': 47,
  'num_followers': 1,
  'tracks': [{'pos': 0,
    'artist_name': 'Missy Elliott',
    'track_uri': 'spotify:track:0UaMYEvWZi0ZqiDOoHU3YI',
    'artist_uri': 'spotify:artist:2wIVse2owClT7go1WT98tk',
    'track_name': 'Lose Control (feat. Ciara & Fat Man Scoop)',
    'album_uri': 'spotify:album:6vV5UrXcfyQD1wu4Qo2I9K',
    'duration_ms': 226863,
    'album_name': 'The Cookbook'},
   {'pos': 1,
    'artist_name': 'Britney Spears',
    'track_uri': 'spotify:track:6I9VzXrHxO9rA9A5euc8Ak',
    'artist_uri': 'spotify:artist:26dSoYclwsYLMAKD3tpOr4',
    'track_name': 'Toxic',
    'album_uri': 'spotify:album:0z7pVBGOD7HCIB7S8eLkLI',
    'duration_ms': 198800,
    'album_name': 'In The Zone'},
   {'pos': 2,
    'artist_name': 'Beyoncé',
    'track_uri': 'spotify:track:0WqIKmW4BTrj3eJFmnCKMv',
    'artist_uri': 'spotify:artist:6vWDO969PvNqNYHIOW5v0m',
    

### extract uniqe track infos

In [13]:
uniq_tracks = set() # set of tuples: [ (artist, album, track) ]
durations = defaultdict(set) # dict of { track_tuple: [duration] }

for p in playlists:
  # list of tuples: [ (artist, album, track, duration) ]
  for t in p['tracks']:
    track_tuple = (t['artist_name'].strip(), t['album_name'].strip(), t['track_name'].strip())
    if track_tuple not in durations:
      uniq_tracks.add(track_tuple)
    durations[track_tuple].add(t['duration_ms'])

len(uniq_tracks), list(uniq_tracks)[:5], durations[list(uniq_tracks)[0]]

(2256502,
 [('Paschalis & Olympians', 'Oi Chryses Epitychies', 'Paradosou Loipon'),
  ('NYCYPCD', 'Splendid Church Life!', 'God Has Called Us For His Purpose'),
  ('Royce Campbell', 'Cocktail Hour', 'Romance Bossa'),
  ('Olga Tanon', 'Siente El Amor', 'Entre La Noche Y El Dia - Remix'),
  ('Gavin Bryars', 'Hommages', 'My First Homage')],
 {213306})

In [15]:
# filter classic tracks
classical = set()

for artist, album, track in list(uniq_tracks):
  album_l = album.lower()
  track_l = track.lower()
  
  if ('op. ' in track_l and 'no. ' in track_l and ('minor' in track_l or 'major' in track_l)):
    classical.add((artist, album, track))
    
  if ('op. ' in album_l and 'no. ' in album_l and ('minor' in album_l or 'major' in album_l)):
    classical.add((artist, album, track))

len(classical), list(classical)[:5]

(9301,
 [('Pyotr Ilyich Tchaikovsky',
   'Tchaikovsky: Piano Concerto No.1 : Piano Concerto No.2',
   'Piano Concerto No. 2 in G Major, Op. 44: I. Allegro brilliante e molto vivace'),
  ('Antonín Dvořák',
   'Dvorak: Symphonies Nos 7,8 & 9',
   "Dvorak: Symphony No. 9 in E Minor, Op. 95, 'From the New World': I. Adagio - Allegro molto"),
  ('Sergei Prokofiev',
   'Jascha Heifetz Violin Concertos - Original Album Classics',
   'Violin Concerto No. 2 in G Minor, Op. 63: III. Allegro ben marcato'),
  ('Frédéric Chopin',
   'Chopin: 57 Mazurkas, Vol. 1',
   'Mazurka in D major, Op. 33. No. 2'),
  ('Ludwig van Beethoven',
   'Beethoven: Complete Symphonies, Vol. 2 (Nos. 5-8)',
   'Symphony No. 5 in C Minor, Op. 67: IV. Allegro')])

In [17]:
len(uniq_tracks) - len(classical)

2247201

In [18]:
uniq_tracks = uniq_tracks - classical
len(uniq_tracks), len(uniq_tracks) == 2_247_201

(2247201, True)

In [19]:
for t in classical:
  durations.pop(t, None)

len(durations.keys())

2247201

In [21]:
durations_mean = { k: np.mean(list(v)) if len(v) > 1 else list(v)[0] for k, v in durations.items() }
durations_std = { k: np.std(list(v)) if len(v) > 1 else 0 for k, v in durations.items() }

### save infos as .csv files

In [ ]:
with open('./data/classical_tracks.csv', 'w') as f:
  writer = csv.writer(f)
  writer.writerow(['artist', 'album', 'track'])
  writer.writerows(classical)

In [22]:
with open('./data/uniq_tracks.csv', 'w') as f:
  writer = csv.writer(f)
  writer.writerow(['artist', 'album', 'track', 'druation_mean', 'duration_std'])
  
  for track in uniq_tracks:
    writer.writerow(list(track) + [durations_mean[track], durations_std[track]])

## YT metadata crawl

### Load Data

In [3]:
df = pd.read_csv('./data/uniq_tracks.csv')
df.head()

,artist,album,track,druation_mean,duration_std
0,Leslie Odom Jr.,Simply Christmas,The Christmas Song,245841.0,0.0
1,Gary Numan,Hope Bleeds,Absolution,299613.0,0.0
2,Lazerhawk,Redline,Dream Machine,207222.0,0.0
3,Benton Blount,Stripped,Wings of Your Love,241875.0,0.0
4,Morrissey,Bona Drag,Lifeguard On Duty,173440.0,0.0


In [4]:
df[df['artist'] == 'Wylie & The Wild West']

,artist,album,track,druation_mean,duration_std
36508,Wylie & The Wild West,Get Wild,Room to Roam,215533.0,0.0
57867,Wylie & The Wild West,Get Wild,I'm Gonna Be a Cowboy,135733.0,0.0
97493,Wylie & The Wild West,Glory Trail,The Grand Roundup,216466.0,0.0
207908,Wylie & The Wild West,Sky Tones,The Yodeling Fool,212253.0,0.0
340522,Wylie & The Wild West,Raven On the Wind,Raven On the Wind,237093.0,0.0
1671644,Wylie & The Wild West,Cattle Call,Cattle Call,244400.0,0.0
2009441,Wylie & The Wild West,Cattle Call,Don't Fence Me In,158866.0,0.0
2157868,Wylie & The Wild West,Rocketbuster,Montana Love Song,165946.0,0.0


### Youtube search

In [ ]:
# def search_youtube(query, max_results=5, format_output=None, download=False, cookie_path=None):
#   cmd = ["yt-dlp"]
  
#   if not download:
#     cmd.append("--skip-download")
  
#   match format_output:
#     case "url":
#       cmd.extend(["--quiet", "--get-id"])
#     case "basic":
#       cmd.extend(["--quiet", "--get-title", "--get-id"])
#     case "detailed":
#       cmd.extend(["--quiet", "--get-title", "--get-id", "--get-duration", "--get-description"])
#     case "json":
#       cmd.extend(["--quiet", "--dump-json"])
  
#   if cookie_path:
#     cmd.extend(["--cookies", cookie_path])
  
#   search_term = f"ytsearch{max_results}:{query}"
#   cmd.append(search_term)
  
#   try:
#     result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    
#     if format_output == "json":
#       data = [ json.loads(line) for line in result.stdout.strip().split('\n') if line ]
#       return data
#     else:
#       return result.stdout
  
#   except subprocess.CalledProcessError as e:
#     print(f"Error: {e}", file=sys.stderr)
#     print(f"Error output: {e.stderr}", file=sys.stderr)
#     return None
#   except Exception as e:
#     print(f"Unexpected error: {e}", file=sys.stderr)
#     return None

In [7]:
test_tracks = df.head(5)
test_tracks

,artist,album,track,druation_mean,duration_std
0,Leslie Odom Jr.,Simply Christmas,The Christmas Song,245841.0,0.0
1,Gary Numan,Hope Bleeds,Absolution,299613.0,0.0
2,Lazerhawk,Redline,Dream Machine,207222.0,0.0
3,Benton Blount,Stripped,Wings of Your Love,241875.0,0.0
4,Morrissey,Bona Drag,Lifeguard On Duty,173440.0,0.0


In [8]:
def search_youtube(
  query:str, 
  max_results:int=5, 
  download:bool=False, 
  cookie_path:str=None
) -> list:
  
  ydl_opts = {
    'format': 'bestaudio/best',
    'quiet': True,
    'no_warnings': True,
    'ignoreerrors': True,
    'extract_flat': 'in_playlist',
    'default_search': f'ytsearch{max_results}',
    'cookies': cookie_path,
    'skip_download': not download,
  }
  
  # Create YoutubeDL object
  with YoutubeDL(ydl_opts) as ydl:
    results = ydl.extract_info(query, download=download)
    
    # Process results
    if 'entries' in results:
      processed_results = []
      
      for entry in results['entries']:
        if entry is None:
          continue
        
        try:
          # get more detailed infos
          video_info = ydl.extract_info(entry['url'], download=False)
          
          # extract relevant infos
          duration_seconds = video_info.get('duration', 0)
          channel_name = video_info.get('uploader', '')
          channel_id = video_info.get('channel_id', '')
          
          result = {
            'id': entry.get('id', ''),
            'title': entry.get('title', ''),
            'url': entry.get('url', ''),
            'channel_name': channel_name,
            'channel_id': channel_id,
            'duration_seconds': duration_seconds,
          }
          
          processed_results.append(result)
        
        except Exception as e:
          print(
            f"Error processing entry {entry.get('id', 'unknown')}: {e}", 
            file=sys.stderr
          )
      
      return processed_results
  
  return []

### Filter search results

In [ ]:
def filter_search_results(result:list, track_info:list, dur_tol=2) -> list:
  artist_i, _, track_i, dur_mean, dur_std = track_info
  dur_mean, dur_std = round(dur_mean / 1000), round(dur_std / 1000)
  
  filtered_results = []
  for r in result:
    artist_i, track_i = artist_i.lower(), track_i.lower()
    
    title_r, uploader_r, duration_r = r.get('title', ''), r.get('channel_name', ''), r.get('duration_seconds', None)
    title_r = title_r.lower() if title_r else ''
    uploader_r = uploader_r.lower() if uploader_r else ''
    
    if 'live' in title_r or 'cover' in title_r or 'remix' in title_r or 'instrumental' in title_r or 'karaoke' in title_r:
      continue
    
    is_right_dur = (
      dur_mean-dur_std <= duration_r <= dur_mean+dur_std
      if dur_std > 0 
      else dur_mean-dur_tol <= duration_r <= dur_mean+dur_tol
    )
    
    score = 0
    if is_right_dur:
      score += 1
    if artist_i in title_r:
      score += 1
    if track_i in title_r:
      score += 1
    if artist_i in uploader_r:
      score += 1
      
    if score > 0:
      r['score'] = score
      filtered_results.append(r)
  
  return filtered_results

### Test pipeline

In [13]:
test_tracks

,artist,album,track,druation_mean,duration_std
0,Leslie Odom Jr.,Simply Christmas,The Christmas Song,245841.0,0.0
1,Gary Numan,Hope Bleeds,Absolution,299613.0,0.0
2,Lazerhawk,Redline,Dream Machine,207222.0,0.0
3,Benton Blount,Stripped,Wings of Your Love,241875.0,0.0
4,Morrissey,Bona Drag,Lifeguard On Duty,173440.0,0.0


In [29]:
total_results = []
length = len(test_tracks)

pbar = tqdm(test_tracks.iterrows(), total=length, dynamic_ncols=True)

for i, (artist, album, track, duration_mean, duration_std) in pbar:
  # query = f'{artist.lower()} {album.lower()} {track.lower()} official lyrics'
  query = f'{artist.lower()} {track.lower()} official lyrics'
  result = search_youtube(
    query, 
    max_results=2, 
    cookie_path='./youtube_cookies.txt',
    download=False
  )
  
  if len(result) > 0:
    pbar.set_description(f"{len(result)} for {artist} - {track}")
    result = filter_search_results(result, (artist, album, track, duration_mean, duration_std))
    result = max(result, key=lambda x: x['score'])
    total_results.append(
      (artist, album, track, duration_mean, duration_std, result)
    )
  
  sleep(0.1)

  0%|          | 0/5 [00:00<?, ?it/s]

2 for Morrissey - Lifeguard On Duty: 100%|██████████| 5/5 [00:23<00:00,  4.73s/it]       


In [17]:
total_results

[('Leslie Odom Jr.',
  'Simply Christmas',
  'The Christmas Song',
  245841.0,
  0.0,
  {'id': 'CUkwsdBzx48',
   'title': 'Leslie Odom Jr. - The Christmas Song (Audio Only)',
   'url': 'https://www.youtube.com/watch?v=CUkwsdBzx48',
   'channel_name': 'Leslie Odom Jr.',
   'channel_id': 'UCIiBME0nYQCeoxi7TbYlCbA',
   'duration_seconds': 246,
   'score': 4}),
 ('Gary Numan',
  'Hope Bleeds',
  'Absolution',
  299613.0,
  0.0,
  {'id': 'ttpmY1Tp0Bo',
   'title': 'Gary Numan - Absolution',
   'url': 'https://www.youtube.com/watch?v=ttpmY1Tp0Bo',
   'channel_name': 'aesthesis',
   'channel_id': 'UCTIcVgUqMPJFrM8aBRvlICA',
   'duration_seconds': 300,
   'score': 3}),
 ('Lazerhawk',
  'Redline',
  'Dream Machine',
  207222.0,
  0.0,
  {'id': 'Qw4SyZNG-0o',
   'title': 'Dream Machine',
   'url': 'https://www.youtube.com/watch?v=Qw4SyZNG-0o',
   'channel_name': 'Lazerhawk - Topic',
   'channel_id': 'UCnIj-rNV30TA1ZLpEcjCFJg',
   'duration_seconds': 207,
   'score': 3}),
 ('Benton Blount',
  'St